# Version 2: A Targeted Second Pass for Plaintiffs' Gender and Race

*Stage 4 of the court-complaint project. Improves the recall of the plaintiffs' gender and race without re-running the extraction, and measures the improvement.*

**Why.** The quality check (version 1) found that the pipeline is reliable when it reports a value, but misses about a quarter of the values a complaint states (estimated recall about 74% for gender and 78% for race). The misses come from **retrieval**, not from misreading: the evidence was in the complaint, but not among the passages the model saw (deep in long complaints, in argument sections, in pronouns next to the name, in group statements such as "Plaintiffs are Black").

**What version 2 does.**
1. **Post-processing, no model calls:** setting values that version 1 removed although their quote was verified (the cue-word check found no cue word) are recovered and marked *for review*.
2. **A targeted second pass:** for each individual plaintiff whose gender or race is still missing, a text search over the **whole complaint** collects the sentences that mention the plaintiff (full name, surname, Mr./Ms. Surname; "Plaintiff" in single-plaintiff complaints; "I"/"my" in pro se complaints) together with a gender or race cue, plus group statements about "Plaintiffs". The model decides from those sentences only, under the coding rules (explicit vs. inferred; the pronoun must refer to this plaintiff; no inference from names, places or national origin, except Spanish-speaking origins for Hispanic/Latino; group statements apply only if they cover this plaintiff). Every value must quote one of the candidate sentences exactly (citation check).
3. **Version-1 results stay unchanged.** Version 2 adds values in separate columns (`gender_v2`, `race_v2`), each with its source and evidence.
4. **Measurement on fresh data:** a new sample of plaintiffs still without a value (recall) and a sample of values found by the second pass (precision), checked by hand, then recall before vs. after.

**Cost and time:** one short model call per plaintiff with a gap (about 500 at most): roughly $1–3 with gpt-4o and a few minutes. Answers are cached, so the run can be resumed or repeated without new calls.

## Setup

In [ ]:
!pip install -q pymupdf openai pandas openpyxl

In [ ]:
import os, re, json, glob, getpass
import pandas as pd
from openai import OpenAI
from google.colab import drive
drive.mount("/content/drive")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key and press Enter: ")
# a key pasted from a document can carry curly quotes or spaces, which break the request headers
key = os.environ["OPENAI_API_KEY"].strip().strip("'\"\u2018\u2019\u201c\u201d ")
bad = [(i, ch) for i, ch in enumerate(key) if not ch.isascii()]
assert not bad, f"The API key contains non-ASCII characters at {bad}: paste it again (plain text)"
os.environ["OPENAI_API_KEY"] = key
client = OpenAI()
test = client.chat.completions.create(model="gpt-4o", max_tokens=5, messages=[{"role": "user", "content": "Say OK"}])
print("API key works:", test.choices[0].message.content)

## Settings

- `FINAL_V1`: the version-1 results after the quality check (`final_results.xlsx`, sheet *Plaintiffs (final)*): upload it to the sorted folder.
- `LIBRARY`: the extraction output for the full library (its *evidence* and *cases* sheets are used for the setting recovery).
- `QC_FILE`: the completed quality-check workbook, used to exclude plaintiffs already checked by hand from the fresh sample.
- `RECALL_V1`: the version-1 recall estimates from the quality check.

In [ ]:
SORTED = "/content/drive/MyDrive/COMPLAINTS_SORTED"
COMPLAINT_FOLDER = os.path.join(SORTED, "complaints")
FINAL_V1 = os.path.join(SORTED, "final_results.xlsx")
LIBRARY = os.path.join(COMPLAINT_FOLDER, "library_analysis.xlsx")
QC_FILE = os.path.join(SORTED, "quality_check_final.xlsx")

OUT_DIR = os.path.join(SORTED, "version2")
os.makedirs(OUT_DIR, exist_ok=True)
CACHE = os.path.join(OUT_DIR, "second_pass_cache.json")
RESULTS_V2 = os.path.join(OUT_DIR, "final_results_v2.xlsx")
REVIEW = os.path.join(OUT_DIR, "second_pass_review.xlsx")
SAMPLE = os.path.join(OUT_DIR, "evaluation_sample_v2.xlsx")

SECOND_PASS_MODEL = "gpt-4o"      # the validated model
WORKERS = 4
RECALL_V1 = {"gender": 0.74, "race": 0.78}

for p in (FINAL_V1, LIBRARY, QC_FILE):
    print(("found   " if os.path.exists(p) else "MISSING ") + p)

## Functions

**Candidate scan** (full text, no model calls): page sentences with line breaks, hyphenation and the court's header stamp removed; references to the plaintiff by priority (name, then "Plaintiff", then "I"); cue words for gender and race; the closest sentences of each kind, with the preceding sentence as context. **Decision rules** for the model, and the **citation check** on its quotes.

**Safeguards in code** (applied to every answer, cached or new): a value is rejected if its quote isn't one of the candidate sentences; if it comes from a group statement limited to some plaintiffs ("the plaintiffs **who** were arrested", "**both**" when there are more than two plaintiffs); or, for gender inferred from a pronoun, if the pronoun stands for someone acting **on** the plaintiff ("**he** informed the plaintiff"). For race, a value is also rejected if the quote describes a **population** ("the rights of African Americans", "the Black residents of Clark County"), puts the race term inside a **quoted title or remark**, states no race, or doesn't refer to the plaintiff at all; for gender, if a pronoun stands next to **several plaintiffs' names** (ambiguous). Rejected values are listed with the reason.

In [ ]:
# ---- Second pass (version 2): candidate scan of the full text, then a model decision ----------------
import re, json, os, time, threading, hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
import fitz
import pandas as pd

GENDER_CUES = re.compile(
    r"\b(he|him|his|himself|she|her|hers|herself|mr|mrs|ms|miss|man|woman|male|female|boy|girl|mother|father|"
    r"son|daughter|sister|brother|wife|husband|grandmother|grandfather|aunt|uncle|niece|nephew|widow|widower|"
    r"nonbinary|non-binary|transgender|they/them)\b\.?", re.I)
RACE_CUES = re.compile(
    r"\b(black|african[- ]american|african|afro[- ]\w+|negro|hispanic|latino|latina|latinx|white|caucasian|"
    r"asian|native american|american indian|indigenous|tribe|tribal|navajo|pacific islander|middle eastern|"
    r"arab|persons? of color|people of color|man of color|woman of color|biracial|mixed[- ]race|multiracial)\b", re.I)
RACE_CATEGORIES = ["Black", "Hispanic/Latino", "White", "Asian", "Native American", "Pacific Islander",
                   "Middle Eastern/North African", "Multiple", "Person of color (unspecified)", "Other (as stated)"]


def page_sentences(pdf_path, max_pages=60):
    """[(page, sentence)] from the complaint's text; line breaks and hyphenation removed."""
    out = []
    with fitz.open(pdf_path) as d:
        for i in range(min(max_pages, len(d))):
            t = d[i].get_text()
            t = re.sub(r"-\n(?=[a-z])", "", t)
            t = re.sub(r"\s+", " ", t)
            t = re.sub(r"Case \S+ Document \S+.*?Page \d+ of \d+( PageID[^ ]* \S+)?", " ", t)   # court header stamp
            for s in re.split(r"(?<=[.!?;])\s+(?=[A-Z0-9\"“(\[])", t):
                s = s.strip()
                if 20 <= len(s) <= 900:
                    out.append((i + 1, s))
    return out


def name_anchors(name, n_plaintiffs, pro_se):
    """References to this plaintiff, by priority: 0 = full name, surname or Mr./Ms. Surname;
    1 = 'Plaintiff' (single-plaintiff complaints only); 2 = 'I'/'my' (pro se complaints only)."""
    name = re.sub(r"\s+", " ", str(name)).strip()
    parts = [p for p in re.split(r"[ ,]+", name) if p and not re.fullmatch(r"(jr|sr|ii|iii|iv)\.?", p, re.I)]
    pats = []
    names = [re.escape(name)] if name else []
    if len(parts) >= 2 and len(parts[-1]) > 2:
        names.append(r"\b" + re.escape(parts[-1]) + r"\b")
    if names:
        pats.append((0, re.compile("|".join(names), re.I)))
    if n_plaintiffs == 1:
        pats.append((1, re.compile(r"\b(the )?plaintiff\b(?!s)", re.I)))
    if pro_se:
        pats.append((2, re.compile(r"\b(I|I'm|my)\b")))
    return pats


GROUP_RE = re.compile(r"\bplaintiffs\b", re.I)


def _distance(s, anchor, cue_re):
    """Smallest distance (in characters) between a reference and a cue word."""
    a = [m.start() for m in anchor.finditer(s)]
    c = [m.start() for m in cue_re.finditer(s)]
    if not a or not c:
        return 10_000
    return min(abs(x - y) for x in a for y in c)


def candidates(sents, anchors, cue_re, n_plaintiffs, limit=20):
    """Sentences with a reference to the plaintiff and a cue word (plus group statements about 'Plaintiffs'
    in multi-plaintiff complaints), ranked by the kind of reference (name first) and by how close the cue
    is to it; each with the preceding sentence as context."""
    found = []
    for k, (pg, s) in enumerate(sents):
        if not cue_re.search(s):
            continue
        best = None
        for prio, rx in anchors:
            if rx.search(s):
                d = _distance(s, rx, cue_re)
                if best is None or (prio, d) < best:
                    best = (prio, d)
        is_group = n_plaintiffs > 1 and GROUP_RE.search(s)
        if best is None and is_group:
            best = (1, _distance(s, GROUP_RE, cue_re))
        if best is None:
            continue
        prev = sents[k - 1][1] if k > 0 and sents[k - 1][0] == pg else ""
        found.append({"page": pg, "sentence": s, "context": prev, "group": bool(is_group),
                      "priority": best[0], "distance": best[1]})
    # the closest sentences of each kind of reference (name, 'Plaintiff', 'I'), so one kind cannot crowd out
    # the others
    out = []
    for prio in (0, 1, 2):
        grp = sorted([c for c in found if c["priority"] == prio], key=lambda c: (c["distance"], c["page"]))
        out += grp[:limit // 2]
    return sorted(out, key=lambda c: (c["priority"], c["distance"]))[:limit]


SYSTEM_PROMPT = """You code a plaintiff's gender and race from sentences of a civil-rights complaint.
Rules:
- Use ONLY the sentences given. Report a value only if a sentence states or clearly indicates it FOR THIS PLAINTIFF.
- Gender: "explicit" if a gender term describes the plaintiff (man, woman, male, female, mother, son, sister, a nonbinary person...); "inferred" if only pronouns (he/she, his/her, himself/herself) or titles (Mr./Ms./Mrs.) refer to the plaintiff. The pronoun must clearly refer to this plaintiff (e.g. right after the plaintiff's name or "Plaintiff"); pronouns about officers, relatives, witnesses or other people do not count. A pronoun counts ONLY if it stands for the plaintiff: if the sentence names the plaintiff as the object of someone else's action ("he informed the plaintiff", "he handcuffed Plaintiff", "she told Mr. X"), the pronoun is that other person. When in doubt, answer "not stated". An explicit gender identity statement ("a nonbinary person") outranks pronouns. Never infer gender from a first name or a name suffix (Jr., Sr.).
- Race: report only a race or ethnicity stated for the plaintiff, as written (e.g. "African American", "Hispanic", "enrolled member of the Navajo Tribe"). Never infer race from a name, a place, a language, a national origin (except Spanish-speaking origins, which indicate Hispanic/Latino), or from descriptions of other people (officers, suspects, decedents, family members, other cases). An allegation of racial profiling without the plaintiff's race is NOT a race, and neither is a statement about a population ("the rights of African Americans", "the Black residents of the county", "like so many African Americans").
- Group statements: a statement about ALL plaintiffs ("Plaintiffs are Black") applies to this plaintiff; a statement about a subset ("the plaintiffs who were arrested") applies only if this plaintiff is in it.
- In a first-person (pro se) complaint, "I"/"my" refers to the plaintiff ("I am a black man").
- The quote must be copied EXACTLY from one sentence (no more than 40 words).
Return JSON only:
{"gender": {"value": "Male|Female|Nonbinary|not stated", "basis": "explicit|inferred|", "quote": "", "page": 0},
 "race": {"value": "as written or not stated", "category": "<one of: %s>|not stated", "quote": "", "page": 0}}""" % ", ".join(RACE_CATEGORIES)


def norm_text(s):
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()


# race: a statement about a population ("rights of African Americans", "the Black residents of Clark County"),
# a quoted title, or a quote with no reference to the plaintiff is not the plaintiff's race
RACE_GENERAL_RE = re.compile(
    r"\b(of|toward|towards|against|for|like|many|other|so many)\s+(the\s+)?(black|blacks|african[- ]americans?|hispanics?|"
    r"latinos?|latinas?|whites?|caucasians?|asians?|native americans?)\b|"
    r"\b(black|african[- ]american|hispanic|latino|latina|white|caucasian|asian)\s+(americans|residents|people|persons|"
    r"community|communities|citizens|neighborhoods?|men|women|individuals|drivers|youths?|families|population)\b", re.I)
QUOTED_RE = re.compile(r"[\"“][^\"”]*\b(black|white|african|hispanic|latino)\b[^\"”]*[\"”]", re.I)
PLAINTIFF_REF_RE = re.compile(r"\b(plaintiffs?|I|I'm|me|my|we|us|our)\b", re.I)


def race_guard(quote, name_rx):
    """Reason to reject a race value, or None."""
    if not RACE_CUES.search(quote):
        return "quote states no race"
    if RACE_GENERAL_RE.search(quote):
        return "describes a population, not the plaintiff"
    if QUOTED_RE.search(quote):
        return "race term inside a quoted title or remark"
    if not (any(rx.search(quote) for rx in name_rx) or PLAINTIFF_REF_RE.search(quote)):
        return "quote does not refer to the plaintiff"
    return None


def other_plaintiffs_rx(names, own):
    """Surnames of the complaint's other plaintiffs (a pronoun next to several plaintiffs is ambiguous)."""
    sur = []
    for n in names:
        if str(n) == str(own):
            continue
        parts = [p for p in re.split(r"[ ,]+", str(n)) if p and not re.fullmatch(r"(jr|sr|ii|iii|iv)\.?", p, re.I)]
        if parts and len(parts[-1]) > 2:
            sur.append(re.escape(parts[-1]))
    return re.compile(r"\b(" + "|".join(sur) + r")\b", re.I) if sur else None


PRONOUN_RE = re.compile(r"\b(he|him|his|himself|she|her|hers|herself)\b", re.I)
# a restrictive clause right after "plaintiffs" ("the plaintiffs who were arrested", no comma), a number, or a
# quantifier limits a group statement to some plaintiffs; "plaintiffs, who are Black" (with a comma) does not
SUBSET_RE = re.compile(r"plaintiffs\s+(in this case\s+)?(who|whom|that|which)\b|\b(two|three|four|five|some|several|"
                       r"certain|one of|most of|many of)\b", re.I)


def verify(answer, cands, anchors=None, n_plaintiffs=1, others_rx=None):
    """Keep a value only if (1) its quote occurs in one of the candidate sentences (citation check);
    (2) a group statement covers ALL plaintiffs (not 'the plaintiffs who were arrested'); and (3) for gender
    inferred from a pronoun, the pronoun comes after the reference to the plaintiff (in 'he informed the
    plaintiff', 'he' is someone else). These checks run on the answer, so they also apply to cached answers."""
    pool = " | ".join(norm_text(c["sentence"]) + " " + norm_text(c["context"]) for c in cands)
    name_rx = [rx for p, rx in (anchors or []) if p == 0]
    all_rx = [rx for p, rx in (anchors or [])]
    out = {}
    for var in ("gender", "race"):
        a = (answer or {}).get(var) or {}
        val = str(a.get("value", "")).strip()
        if not val or val.lower() == "not stated":
            continue
        quote = str(a.get("quote", ""))
        q = norm_text(quote)
        if len(q) < 8 or q not in pool:
            out[var] = {"rejected": True, "reason": "quote not found in the candidate sentences", **a}
            continue
        names_plaintiff = any(rx.search(quote) for rx in name_rx)
        if n_plaintiffs > 1 and not names_plaintiff and GROUP_RE.search(quote):
            subset = SUBSET_RE.search(quote) or (re.search(r"\bboth\b", quote, re.I) and n_plaintiffs != 2)
            if subset:
                out[var] = {"rejected": True, "reason": "group statement limited to some plaintiffs", **a}
                continue
        if var == "race":
            why = race_guard(quote, name_rx)
            if why:
                out[var] = {"rejected": True, "reason": why, **a}
                continue
        if var == "gender" and str(a.get("basis", "")).lower() == "inferred" and others_rx is not None \
                and others_rx.search(quote):
            out[var] = {"rejected": True, "reason": "pronoun next to several plaintiffs (ambiguous)", **a}
            continue
        if var == "gender" and str(a.get("basis", "")).lower() == "inferred":
            refs = [m.start() for rx in all_rx for m in rx.finditer(quote)] + \
                   ([m.start() for m in GROUP_RE.finditer(quote)] if n_plaintiffs > 1 else [])
            prons = [m.start() for m in PRONOUN_RE.finditer(quote)]
            title = re.search(r"\b(mr|mrs|ms)\.?\s+\w+", quote, re.I) and names_plaintiff
            if prons and refs and not title and min(prons) < min(refs) and \
                    not any(p > min(refs) for p in prons):
                out[var] = {"rejected": True, "reason": "pronoun refers to someone acting on the plaintiff", **a}
                continue
        out[var] = a
    return out


**Running the second pass:** one cached model call per plaintiff with a gap, in parallel.

In [ ]:
# ---- Running the second pass ---------------------------------------------------------------------------
def ask_model(name, needs, cands, n_plaintiffs, pro_se):
    """One model call per plaintiff: the candidate sentences in, a JSON decision out."""
    lines = []
    for c in cands:
        ctx = f"(previous sentence: {c['context'][:300]}) " if c["context"] else ""
        lines.append(f"[p. {c['page']}] {ctx}{c['sentence']}")
    user = (f"Plaintiff: {name}\nNumber of plaintiffs in the complaint: {n_plaintiffs}\n"
            f"Self-written (pro se) complaint: {'yes' if pro_se else 'no'}\n"
            f"Code only: {', '.join(needs)} (answer 'not stated' for anything else).\n\n"
            "Sentences:\n" + "\n".join(lines))
    resp = client.chat.completions.create(
        model=SECOND_PASS_MODEL, temperature=0, response_format={"type": "json_object"},
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}])
    return json.loads(resp.choices[0].message.content)


def _cache_key(row, needs):
    return hashlib.md5(f"{row['source_file']}|{row['name_final']}|{','.join(needs)}|v2b".encode()).hexdigest()


def second_pass(plaintiffs, folder, cache_path, workers=4):
    """For each individual plaintiff with gender or race missing: candidate sentences from the full text,
    then a model decision (cached, so the cell can be re-run and resumed). Returns one row per plaintiff and
    variable."""
    cache = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
    lock = threading.Lock()
    todo = plaintiffs[(plaintiffs.plaintiff_type == "person") & (plaintiffs.exclude.fillna("") == "") &
                      (plaintiffs.gender_final.isna() | plaintiffs.race_final.isna())]
    n_by_file = plaintiffs.groupby("source_file").size()
    names_by_file = plaintiffs.groupby("source_file")["name_final"].apply(list).to_dict()
    sent_cache = {}

    def work(row):
        needs = [v for v, col in (("gender", "gender_final"), ("race", "race_final")) if pd.isna(row[col])]
        path = os.path.join(folder, row["source_file"])
        if not os.path.exists(path):
            return [{"variable": v, "status": "file not found"} for v in needs]
        if row["source_file"] not in sent_cache:
            sent_cache[row["source_file"]] = page_sentences(path)
        sents = sent_cache[row["source_file"]]
        n_pl = int(n_by_file.get(row["source_file"], 1))
        anchors = name_anchors(row["name_final"], n_pl, bool(row["pro_se"]))
        other_rx = other_plaintiffs_rx(names_by_file.get(row["source_file"], []), row["name_final"])
        cands, have = [], {}
        for v in needs:
            cv = candidates(sents, anchors, GENDER_CUES if v == "gender" else RACE_CUES, n_pl)
            have[v] = bool(cv)
            cands += [c for c in cv if c["sentence"] not in {x["sentence"] for x in cands}]
        ask = [v for v in needs if have[v]]
        out = [{"variable": v, "status": "no candidate sentences"} for v in needs if not have[v]]
        if not ask:
            return out
        key = _cache_key(row, ask)
        with lock:
            answer = cache.get(key)
        if answer is None:
            answer = ask_model(row["name_final"], ask, cands, n_pl, bool(row["pro_se"]))
            with lock:
                cache[key] = answer
        checked = verify(answer, cands, anchors, n_pl, other_rx)
        for v in ask:
            a = checked.get(v)
            if a is None:
                out.append({"variable": v, "status": "not stated"})
            elif a.get("rejected"):
                out.append({"variable": v, "status": f"rejected: {a.get('reason')}", "value": a.get("value"),
                            "quote": a.get("quote"), "page": a.get("page")})
            else:
                out.append({"variable": v, "status": "found", "value": a.get("value"), "basis": a.get("basis", ""),
                            "category": a.get("category", ""), "quote": a.get("quote"), "page": a.get("page")})
        return out

    rows = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(work, r): r for _, r in todo.iterrows()}
        for k, fut in enumerate(as_completed(futures), start=1):
            r = futures[fut]
            try:
                res = fut.result()
            except Exception as e:
                res = [{"variable": "?", "status": f"error: {str(e)[:120]}"}]
            for x in res:
                rows.append({"source_file": r["source_file"], "name": r["name_final"], "pro_se": r["pro_se"], **x})
            if k % 25 == 0:
                with lock:
                    json.dump(cache, open(cache_path, "w"))
                print(f"  {k}/{len(futures)} plaintiffs")
    json.dump(cache, open(cache_path, "w"))
    return pd.DataFrame(rows)


def race_category(value, category):
    """Standard race category from the model's answer (the category if valid, else a simple mapping)."""
    v = str(value).lower()
    if re.search(r"\bnot white\b|non-?white", v):
        return "Person of color (unspecified)"
    if category in RACE_CATEGORIES:
        return category
    for pat, cat in [(r"black|african|negro", "Black"), (r"hispanic|latin", "Hispanic/Latino"),
                     (r"white|caucasian", "White"), (r"asian", "Asian"),
                     (r"native|indian|indigenous|tribe|navajo", "Native American"),
                     (r"pacific", "Pacific Islander"), (r"middle eastern|arab", "Middle Eastern/North African"),
                     (r"mixed|biracial|multiracial", "Multiple"), (r"of color", "Person of color (unspecified)")]:
        if re.search(pat, v):
            return cat
    return "Other (as stated)"

**Version-2 dataset, setting recovery, evaluation samples and summary tables.**

In [ ]:
# ---- Version 2 dataset: version-1 values unchanged, second-pass values added with their evidence ---------
def build_v2(plaintiffs, results):
    pv2 = plaintiffs.copy()
    for col in ["gender_v2", "race_v2", "gender_v2_source", "race_v2_source", "gender_v2_evidence", "race_v2_evidence"]:
        pv2[col] = None
    pv2["gender_v2"] = pv2["gender_final"]
    pv2["race_v2"] = pv2["race_final"]
    pv2.loc[pv2.gender_final.notna(), "gender_v2_source"] = "version 1"
    pv2.loc[pv2.race_final.notna(), "race_v2_source"] = "version 1"
    found = results[results.status == "found"]
    for _, r in found.iterrows():
        ix = pv2.index[(pv2.source_file == r.source_file) & (pv2.name_final == r["name"])]
        ev = f'p. {r.page}: "{r.quote}"'
        if r.variable == "gender" and str(r.value) in ("Male", "Female", "Nonbinary"):
            pv2.loc[ix, ["gender_v2", "gender_v2_source", "gender_v2_evidence"]] = \
                [r.value, f"second pass ({r.basis or 'n/a'})", ev]
        elif r.variable == "race":
            pv2.loc[ix, ["race_v2", "race_v2_source", "race_v2_evidence"]] = \
                [race_category(r.value, r.category), f"second pass (as written: {r.value})", ev]
    return pv2


def recover_settings(evidence, cases):
    """Setting values the version-1 check removed although their quote was found on the cited page (the check
    found no cue word): recovered, marked 'for review'. No model calls."""
    e = evidence[(evidence.field.isin(["primary_setting", "primary_setting_detail"])) & (evidence.status == "failed") &
                 (evidence.problems.fillna("").str.strip() == "quote does not contain the value")]
    rec = e.pivot_table(index="source_file", columns="field", values="value", aggfunc="first").reset_index()
    rec = rec.rename(columns={"primary_setting": "setting_recovered", "primary_setting_detail": "setting_detail_recovered"})
    out = cases.merge(rec, on="source_file", how="left")
    out["setting_v2"] = out["setting"].where(out["setting"].notna(), out.get("setting_recovered"))
    out["setting_detail_v2"] = out["setting_detail"].where(out["setting_detail"].notna(), out.get("setting_detail_recovered"))
    out["setting_v2_status"] = None
    out.loc[out["setting"].notna(), "setting_v2_status"] = "version 1 (verified)"
    out.loc[out["setting"].isna() & out["setting_v2"].notna(), "setting_v2_status"] = "recovered: quote verified, no cue word (for review)"
    return out


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * (p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5 / d
    return (c - h, c + h)

In [ ]:
# ---- Fresh evaluation samples ----------------------------------------------------------------------------
def draw_samples(pv2, checked_before, n_recall=30, n_precision=20, seed=2027):
    """(a) recall: plaintiffs still without a value after version 2 (half pro se, half represented), excluding
    those checked in the first quality check; (b) precision: values found by the second pass."""
    people = pv2[(pv2.plaintiff_type == "person") & (pv2.exclude.fillna("") == "")].copy()
    people["key"] = people.source_file + "|" + people.name_final.astype(str)
    people = people[~people.key.isin(checked_before)]
    rows = []
    for var in ("gender", "race"):
        still = people[people[f"{var}_v2"].isna()]
        for ps in (True, False):
            g = still[still.pro_se == ps]
            take = g.sample(min(n_recall // 2, len(g)), random_state=seed)
            for _, r in take.iterrows():
                rows.append({"check": "recall", "variable": var, "case_name": r.case_name, "source_file": r.source_file,
                             "name": r.name_final, "pro_se": ps, "value_v2": "", "evidence_v2": "",
                             "stated_in_complaint": "", "value_correct": "", "correct_value": "", "note": ""})
        found = people[people[f"{var}_v2_source"].fillna("").str.startswith("second pass")]
        take = found.sample(min(n_precision, len(found)), random_state=seed)
        for _, r in take.iterrows():
            rows.append({"check": "precision", "variable": var, "case_name": r.case_name, "source_file": r.source_file,
                         "name": r.name_final, "pro_se": r.pro_se, "value_v2": r[f"{var}_v2"],
                         "evidence_v2": r[f"{var}_v2_evidence"], "stated_in_complaint": "", "value_correct": "", "correct_value": "", "note": ""})
    return pd.DataFrame(rows)


def evaluate(pv2, sample, before):
    """Recall before and after the second pass (stratified by pro se), and the second pass's precision."""
    people = pv2[(pv2.plaintiff_type == "person") & (pv2.exclude.fillna("") == "")]
    s = sample.fillna("")
    # only rows you filled in count (so a partly completed sample gives an unbiased, if less precise, estimate)
    filled = lambda col: s[col].str.strip().str.lower().isin(["yes", "no"])
    s_rec = s[(s.check == "recall") & filled("stated_in_complaint")]
    s_pre = s[(s.check == "precision") & filled("value_correct")]
    rows = []
    for var in ("gender", "race"):
        auto_v1 = people[f"{var}_pipeline"].notna().sum()
        found2 = people[f"{var}_v2_source"].fillna("").str.startswith("second pass").sum()
        est_missed, var_sum, n_still = 0.0, 0.0, 0
        for ps in (True, False):
            sub = s_rec[(s_rec.variable == var) & (s_rec.pro_se.astype(str).str.lower() == str(ps).lower())]
            n = len(sub)
            k = (sub.stated_in_complaint.str.lower() == "yes").sum()
            N = int((people[f"{var}_v2"].isna() & (people.pro_se == ps)).sum())
            n_still += N
            if n:
                p = k / n
                est_missed += N * p
                var_sum += N * N * p * (1 - p) / n
        # values the pipeline never found but that were added by hand in the first quality check are known misses
        manual = int((people[f"{var}_pipeline"].isna() & people[f"{var}_final"].notna()).sum())
        est_missed += manual
        found_after = auto_v1 + found2
        rec_after = found_after / (found_after + est_missed) if found_after + est_missed else float("nan")
        half = 1.96 * var_sum ** 0.5
        rec_lo = found_after / (found_after + est_missed + half)
        rec_hi = found_after / (found_after + max(manual, est_missed - half))
        prec = s_pre[s_pre.variable == var]
        kp = (prec.value_correct.str.lower() == "yes").sum()
        lo, hi = wilson(kp, len(prec))
        n_rec = int((s_rec.variable == var).sum())
        rows.append({"variable": var, "recall_rows_checked": n_rec, "recall_v1": before[var], "found_by_second_pass": int(found2),
                     "still_not_stated": n_still, "added_by_hand_in_qc": manual, "estimated_still_missed": round(est_missed),
                     "recall_v2": round(rec_after, 2), "recall_v2_ci": f"{rec_lo:.0%}-{rec_hi:.0%}",
                     "second_pass_precision": f"{kp} of {len(prec)}" + (f" ({kp/len(prec):.0%}; 95% CI {lo:.0%}-{hi:.0%})" if len(prec) else "")})
    return pd.DataFrame(rows)

In [ ]:
def summary_tables(pv2):
    """Stated shares and distributions, version 1 vs. version 2 (individual plaintiffs only)."""
    people = pv2[(pv2.plaintiff_type == "person") & (pv2.exclude.fillna("") == "")]
    stated = pd.DataFrame([{
        "variable": var, "stated_v1": round(people[f"{var}_final"].notna().mean(), 3),
        "stated_v2": round(people[f"{var}_v2"].notna().mean(), 3),
        "stated_v2_represented": round(people[people.pro_se == False][f"{var}_v2"].notna().mean(), 3),
        "stated_v2_pro_se": round(people[people.pro_se == True][f"{var}_v2"].notna().mean(), 3)}
        for var in ("gender", "race")])
    def dist(col):
        v = people[col].fillna("not stated")
        t = v.value_counts().rename_axis(col).reset_index(name="plaintiffs")
        st = t.loc[t[col] != "not stated", "plaintiffs"].sum()
        t["share_of_stated"] = [round(n / st, 3) if c != "not stated" else None for c, n in zip(t[col], t.plaintiffs)]
        return t
    return stated, dist("gender_v2"), dist("race_v2")

**F1 and accuracy:** precision, recall, F1 and accuracy of the overall automated pipeline, with bootstrap intervals. Set `PRECISION_V1` from the gold-set validation (`03_Validation.ipynb`, the share of stated values that were correct).

In [ ]:
# ---- F1 and accuracy (overall pipeline), with bootstrap intervals ----------------------------------------------
# Precision of the version-1 values comes from the gold-set validation (03_Validation.ipynb, "Key metrics"): the
# share of stated values that were correct. Set these from your validation output.
PRECISION_V1 = {"gender": 1.00, "race": 0.95}
N_BOOT = 2000


def overall_metrics(people, var, auto_mask, added_mask, still_col, sample, precision_col, recall_col="stated_in_complaint",
                    p_v1=None, n_boot=N_BOOT, seed=7):
    """Precision, recall, F1 and accuracy of the automated pipeline for one variable, with 95% bootstrap intervals.
    - precision: version-1 values (precision from the gold set) and added values (precision sample), weighted;
    - recall: values found automatically / (found + estimated misses); misses = values added by hand earlier +
      stratified estimate from the recall sample (pro se / represented);
    - accuracy: share of all individual plaintiffs whose value is right, counting a correct "not stated" as right
      (errors = estimated misses + estimated wrong values). It depends on how often the value is absent, so read
      it alongside F1.
    The bootstrap resamples the recall rows within each stratum and the precision rows; the version-1 precision is
    held fixed (its own uncertainty comes from the gold set)."""
    import numpy as np
    rng = np.random.default_rng(seed)
    p_v1 = PRECISION_V1[var] if p_v1 is None else p_v1
    s = sample.fillna("")
    s = s[s.variable == var]
    yn = lambda col: s[col].str.strip().str.lower().isin(["yes", "no"])
    rec = s[(s.check == "recall") & yn(recall_col)]
    pre = s[(s.check == "precision") & yn(precision_col)]
    auto = int(auto_mask.sum())
    n_add = int(added_mask.sum())
    n_v1 = auto - n_add
    manual = int((people[still_col].notna() & ~auto_mask).sum())
    N_people = len(people)
    strata = []
    for ps in (True, False):
        sub = rec[rec.pro_se.astype(str).str.lower() == str(ps).lower()]
        y = (sub[recall_col].str.lower() == "yes").to_numpy()
        N = int((people[still_col].isna() & (people.pro_se == ps)).sum())
        strata.append((y, N))
    ypre = (pre[precision_col].str.lower() == "yes").to_numpy()

    def compute(strata_y, ypre_s):
        missed = manual + sum(N * (y.mean() if len(y) else 0.0) for y, (_, N) in zip(strata_y, strata))
        p_add = ypre_s.mean() if len(ypre_s) else 1.0
        P = (n_v1 * p_v1 + n_add * p_add) / auto if auto else float("nan")
        R = auto / (auto + missed) if auto + missed else float("nan")
        F1 = 2 * P * R / (P + R) if P + R else float("nan")
        wrong = n_v1 * (1 - p_v1) + n_add * (1 - p_add)
        A = 1 - (missed + wrong) / N_people
        return P, R, F1, A

    point = compute([y for y, _ in strata], ypre)
    draws = []
    for _ in range(n_boot):
        sy = [rng.choice(y, size=len(y), replace=True) if len(y) else y for y, _ in strata]
        sp = rng.choice(ypre, size=len(ypre), replace=True) if len(ypre) else ypre
        draws.append(compute(sy, sp))
    draws = np.array(draws)
    lo, hi = np.nanpercentile(draws, 2.5, axis=0), np.nanpercentile(draws, 97.5, axis=0)
    names = ["precision", "recall", "F1", "accuracy"]
    row = {"variable": var}
    for i, n in enumerate(names):
        row[n] = round(point[i], 3)
        row[f"{n}_95ci"] = f"{lo[i]:.2f}-{hi[i]:.2f}"
    row.update({"values_found_automatically": auto, "of_which_added_by_second_pass": n_add,
                "recall_rows": int(sum(len(y) for y, _ in strata)), "precision_rows": int(len(ypre))})
    return row

## Step 1: Load the Version-1 Results

Individual plaintiffs only (organizations, estates and the excluded file are left out of the second pass).

In [ ]:
pv1 = pd.read_excel(FINAL_V1, sheet_name="Plaintiffs (final)")
people = pv1[(pv1.plaintiff_type == "person") & (pv1.exclude.fillna("") == "")]
print(f"{len(pv1)} plaintiff records, {len(people)} individual plaintiffs")
print("gender missing:", int(people.gender_final.isna().sum()), "| race missing:", int(people.race_final.isna().sum()))

## Step 2: Test on a Few Plaintiffs

Runs the second pass on 10 plaintiffs and shows what it found, with the quotes. Check that the values and quotes make sense before the full run.

In [ ]:
test = second_pass(pv1[pv1.index.isin(people[people.gender_final.isna() | people.race_final.isna()].head(10).index)],
                   COMPLAINT_FOLDER, CACHE, workers=WORKERS)
display(test.reindex(columns=["name", "variable", "status", "value", "basis", "quote", "page"]))

## Step 3: The Full Second Pass

In [ ]:
results = second_pass(pv1, COMPLAINT_FOLDER, CACHE, workers=WORKERS)
print(results.status.value_counts().to_string())
results.to_csv(os.path.join(OUT_DIR, "second_pass_results.csv"), index=False)

## Step 4: Review the New Values (optional, quick)

Writes `second_pass_review.xlsx`: every value the second pass found, with its quote and page. Skim it: set `accept` to **no** for any value that is wrong (e.g. the quote describes someone else); the next step leaves those out. Your decisions are kept when the cell is run again: values found since the file was written are added to it, and existing rows are left as they are.

In [ ]:
found = results[results.status == "found"].copy()
cols = ["source_file", "name", "pro_se", "variable", "value", "basis", "category", "quote", "page", "accept"]
found["accept"] = "yes"
found = found.reindex(columns=cols)
if os.path.exists(REVIEW):
    # keep your decisions; add any values found since the file was written (e.g. after a re-run)
    old = pd.read_excel(REVIEW, dtype=str).fillna("")
    key = lambda d: d.source_file.astype(str) + "|" + d["name"].astype(str) + "|" + d.variable.astype(str)
    new = found[~key(found).isin(set(key(old)))]
    merged = pd.concat([old, new.astype(str)], ignore_index=True)
    merged.to_excel(REVIEW, index=False)
    print(f"Review file kept ({len(old)} rows with your decisions); {len(new)} new values added -> {REVIEW}")
else:
    found.to_excel(REVIEW, index=False)
    print(f"{len(found)} new values -> {REVIEW}")
print(found.groupby("variable").size().to_string())

## Step 5: Build the Version-2 Dataset

Version-1 values are kept as they are; second-pass values fill gaps only, with their source and evidence. **The review file decides:** values marked `accept` = yes are used (including any you restored after a safeguard rejected them), values marked no are left out. Setting values recovered from the version-1 evidence are added to the cases table, marked *for review*.

In [ ]:
# the review file is the final word: values marked yes are kept (including any you restored after a safeguard
# rejected them); values marked no are left out
rev = pd.read_excel(REVIEW, dtype=str).fillna("")
keep = rev[rev.accept.str.lower() == "yes"].copy()
keep["status"] = "found"
print(f"Values kept from the review: {len(keep)} ({keep.groupby('variable').size().to_dict()}); "
      f"left out: {int((rev.accept.str.lower() == 'no').sum())}")
pv2 = build_v2(pv1, keep)

lib = pd.read_excel(LIBRARY, sheet_name=None)
cases_v2 = recover_settings(lib["evidence"], lib["cases"])
stated, g_tab, r_tab = summary_tables(pv2)
with pd.ExcelWriter(RESULTS_V2) as w:
    stated.to_excel(w, sheet_name="Summary v2", index=False)
    g_tab.to_excel(w, sheet_name="Summary v2", index=False, startrow=len(stated) + 3)
    r_tab.to_excel(w, sheet_name="Summary v2", index=False, startrow=len(stated) + len(g_tab) + 6)
    pv2.to_excel(w, sheet_name="Plaintiffs v2", index=False)
    keep.to_excel(w, sheet_name="Second pass results", index=False)
    cases_v2.to_excel(w, sheet_name="Cases v2 (settings)", index=False)
print("Saved:", RESULTS_V2)
display(stated)
print(cases_v2.setting_v2_status.value_counts(dropna=False).to_string())

## Step 6: Draw the Fresh Evaluation Samples

Writes `evaluation_sample_v2.xlsx` (never overwritten once it exists):
- **recall rows** (`check` = recall): 30 plaintiffs per variable still without a value after version 2 (15 pro se, 15 represented), excluding the plaintiffs checked in the first quality check. Search each complaint as before: `stated_in_complaint` yes/no; if yes, the value in `correct_value` and the quote and page in `note`.
- **precision rows** (`check` = precision): 20 values per variable found by the second pass, with their evidence. Check each against the complaint: `value_correct` yes/no; if no, the right value (or *not stated*) in `correct_value`.

In [ ]:
qc = pd.read_excel(QC_FILE, sheet_name="Not stated sample", dtype=str).fillna("")
checked_before = set(qc.source_file + "|" + qc.name_clean)
if os.path.exists(SAMPLE):
    print("Sample exists and was NOT overwritten:", SAMPLE)
else:
    sample = draw_samples(pv2, checked_before)
    sample.to_excel(SAMPLE, index=False)
    print(sample.groupby(["check", "variable"]).size().to_string(), "->", SAMPLE)

## Step 7: Recall Before and After, and the Second Pass's Precision

Run after filling in the evaluation sample.

In [ ]:
sample = pd.read_excel(SAMPLE, dtype=str).fillna("")
ev = evaluate(pv2, sample, RECALL_V1)
display(ev)
for _, r in ev.iterrows():
    print(f"{r.variable}: estimated recall {r.recall_v1:.0%} (version 1) -> {r.recall_v2:.0%} "
          f"(version 2; 95% CI {r.recall_v2_ci}); "
          f"second pass found {r.found_by_second_pass} values, precision {r.second_pass_precision}")

## Step 7b: F1 and Accuracy (Version 2)

Precision, recall, **F1** (the harmonic mean of precision and recall) and **accuracy** of the overall automated pipeline (version 1 plus the second pass), with 95% bootstrap intervals from the evaluation sample.

- **Precision** combines the version-1 values (precision from the gold set, `PRECISION_V1`) and the second-pass additions (precision sample).
- **Accuracy** counts every individual plaintiff, including correct "not stated" answers, so it depends on how often a value is absent (for race, nearly half the time): read it alongside F1, which ignores correct absences.

In [ ]:
metric_rows = []
people_m = pv2[(pv2.plaintiff_type == "person") & (pv2.exclude.fillna("") == "")]
for var in ("gender", "race"):
    added = people_m[f"{var}_v2_source"].fillna("").str.startswith("second pass")
    auto = people_m[f"{var}_pipeline"].notna() | added
    metric_rows.append(overall_metrics(people_m, var, auto, added, f"{var}_v2", sample, "value_correct"))
metrics_v2 = pd.DataFrame(metric_rows)
display(metrics_v2)

## Step 8: The Final Dataset

Applies what the evaluation found, so the published dataset is as complete and correct as the checks allow:
- **recall rows marked yes:** the value you found is added (source: *evaluation, by hand*);
- **precision rows marked no:** the second-pass value is removed, or replaced by your `correct_value`;
- **organizations** missed by the name filter, **misread or truncated names** (corrected from the complaints), and **document problems** found during the evaluations (defendants' filings selected as complaints) are recorded.

Writes `final_results_v2_final.xlsx`: the summary tables, the evaluation results, and every plaintiff with version-1, version-2 and final values and their sources. Version-1 and version-2 columns are left unchanged.

In [ ]:
FINAL = os.path.join(OUT_DIR, "final_results_v2_final.xlsx")
# Case-level corrections (organizations listed as plaintiffs, misread names, files that are not complaints) are kept
# in a private file next to the results, not in the notebook: they name individuals. Without it, they are skipped.
OVERRIDES = os.path.join(OUT_DIR, "private_overrides.json")
_ov = json.load(open(OVERRIDES)) if os.path.exists(OVERRIDES) else {}
if not _ov:
    print("private_overrides.json not found: case-level corrections skipped")
ORGANIZATIONS = set(_ov.get("organizations", []))
NAME_FIXES = {(f, old): new for f, old, new in _ov.get("name_fixes", [])}   # keyed by file and name


def fix_names(df, name_col):
    """Apply NAME_FIXES to a table with source_file and a name column."""
    df = df.copy()
    for (f, old), new in NAME_FIXES.items():
        df.loc[(df.source_file == f) & (df[name_col] == old), name_col] = new
    return df
CASE_FLAGS = _ov.get("case_flags", {})   # files that turned out not to be complaints

pf = pv2.copy()
for var in ("gender", "race"):
    pf[f"{var}_final_v2"] = pf[f"{var}_v2"]
    pf[f"{var}_final_v2_source"] = pf[f"{var}_v2_source"]
pf.loc[pf.name_final.isin(ORGANIZATIONS), "plaintiff_type"] = "organization"
pf = fix_names(pf, "name_final")
pf["case_flag"] = pf.source_file.map(CASE_FLAGS).fillna("")

ev_rows = pd.read_excel(SAMPLE, dtype=str).fillna("")
ev_rows = fix_names(ev_rows, "name")
changes = 0
for _, r in ev_rows.iterrows():
    ix = pf.index[(pf.source_file == r.source_file) & (pf.name_final == r["name"])]
    var = r.variable
    if r.check == "recall" and r.stated_in_complaint.strip().lower() == "yes" and r.correct_value.strip():
        val = race_category(r.correct_value, r.correct_value) if var == "race" else r.correct_value.strip()
        pf.loc[ix, [f"{var}_final_v2", f"{var}_final_v2_source"]] = [val, "evaluation, by hand"]
        changes += len(ix)
    elif r.check == "precision" and r.value_correct.strip().lower() == "no":
        cv = r.correct_value.strip()
        new = None if (not cv or cv.lower() == "not stated") else (race_category(cv, cv) if var == "race" else cv)
        pf.loc[ix, [f"{var}_final_v2", f"{var}_final_v2_source"]] = [new, "evaluation, by hand (second-pass value rejected)"]
        changes += len(ix)
print("values changed by the evaluation:", changes)

people_f = pf[(pf.plaintiff_type == "person") & (pf.exclude.fillna("") == "")]
stated_f = pd.DataFrame([{
    "variable": var, "stated_v1": round(people_f[f"{var}_final"].notna().mean(), 3),
    "stated_v2": round(people_f[f"{var}_v2"].notna().mean(), 3),
    "stated_final": round(people_f[f"{var}_final_v2"].notna().mean(), 3),
    "stated_final_represented": round(people_f[people_f.pro_se == False][f"{var}_final_v2"].notna().mean(), 3),
    "stated_final_pro_se": round(people_f[people_f.pro_se == True][f"{var}_final_v2"].notna().mean(), 3)}
    for var in ("gender", "race")])
def dist(col):
    v = people_f[col].fillna("not stated")
    t = v.value_counts().rename_axis(col).reset_index(name="plaintiffs")
    st = t.loc[t[col] != "not stated", "plaintiffs"].sum()
    t["share_of_stated"] = [round(n / st, 3) if c != "not stated" else None for c, n in zip(t[col], t.plaintiffs)]
    return t
g_f, r_f = dist("gender_final_v2"), dist("race_final_v2")
with pd.ExcelWriter(FINAL) as w:
    stated_f.to_excel(w, sheet_name="Summary", index=False)
    g_f.to_excel(w, sheet_name="Summary", index=False, startrow=len(stated_f) + 3)
    r_f.to_excel(w, sheet_name="Summary", index=False, startrow=len(stated_f) + len(g_f) + 6)
    ev.to_excel(w, sheet_name="Evaluation", index=False)
    metrics_v2.to_excel(w, sheet_name="F1 and accuracy", index=False)
    pf.to_excel(w, sheet_name="Plaintiffs (final)", index=False)
print(f"{len(people_f)} individual plaintiffs | saved: {FINAL}")
display(stated_f)
display(r_f)

# Version 2.1: Improving Race Recall

The version-2 evaluation estimated race recall at 81% (95% CI 70–95%). The misses had specific causes, and version 2.1 addresses each, **for race only** (gender values and all earlier results are left as they are):

| Miss in the evaluation | Change in version 2.1 |
|---|---|
| Group statements inside population sentences: "minority groups such as plaintiffs, **who are blacks**" | a race term **attached** to "plaintiff(s)", the name or "I" is accepted, even in a sentence about a population; "plaintiffs, who are…" counts in single-plaintiff templates too |
| Indirect statements: "the **racial minority** of Plaintiff X", "Plaintiff is not white" | coded **Person of color (unspecified)** |
| Self-descriptions buried in long filings: "I am a Black Freedmen" (Moore, 98 pages) | candidate sentences where the race term is attached to the plaintiff rank first; a larger limit (30) for filings over 30 pages |
| Forms and court records: "Race: Black" next to the plaintiff's name | "Race: X" fields on pages that name the plaintiff are candidates, and count |
| Partly scanned complaints (e.g., only page 1 had text) | a **per-page OCR screen** finds files where most pages have no text, and OCRs them |
| Slurs aimed at the plaintiff, "drive while black" | counted as **inferred** when `INCLUDE_INDIRECT = True` (set False for a strict rule) |

The safeguards against population statements, subsets of plaintiffs and descriptions of other people remain. New model calls are made only for plaintiffs whose race is still missing after Step 8 (about 300 short calls; roughly $1–2), cached in a separate file. Then: a review file, the version-2.1 dataset, a fresh evaluation sample (larger for represented plaintiffs, so the interval is narrower), and recall before vs. after.

Run Steps 1–8 first in the same session (cached, free), since version 2.1 starts from Step 8's final dataset (`pf`).

**Version-2.1 functions:** the attachment rule, the race candidate scan (with forms and the long-filing limit), the race prompt and safeguards, the race pass, the per-page OCR screen, and the fresh sample and evaluation.

In [ ]:
# ---- Version 2.1: race recall ------------------------------------------------------------------------------
# Changes for race only, informed by the misses in the version-2 evaluation. The gender values and every
# earlier result are left as they are. New model calls only for plaintiffs whose race is still missing.
import sys, subprocess, shutil

INCLUDE_INDIRECT = True   # count indirect statements about the plaintiff (e.g. "drive while black", a racial
                          # slur aimed at the plaintiff) as inferred race; set False for a strict rule

RACE_TERMS = (r"(blacks?|african[- ]americans?|african|negro|hispanics?|latinos?|latinas?|latinx|whites?|caucasians?|"
              r"asians?|native americans?|american indians?|indigenous|navajo|pacific islanders?|middle eastern|arabs?|"
              r"biracial|mixed[- ]race|multiracial|persons? of color|people of color|man of color|woman of color|"
              r"non-?white|racial minority|racial/ethnic minority|minority)")
RACE_CUES_21 = re.compile(r"\b" + RACE_TERMS + r"\b|\btrib(e|al)\b|\brace\s*[:\-]", re.I)
FORM_RE = re.compile(r"\brace\s*[:\-]?\s*(black|white|hispanic|latino|asian|african[- ]american|native american|"
                     r"american indian|pacific islander|other)\b", re.I)
HEADER_RE_21 = re.compile(r"Case\s+\S+\s+Document\s+\S+.*?Page\s+\d+\s+of\s+\d+(\s+PageID[^\n]*)?", re.I | re.S)
_AFTER = re.compile(r"^\W{0,3}((,?\s*(who|whom|which)\s+(is|are|was|were))|(\s*(is|are|was|were|am)\b)|,)\s*"
                    r"(not\s+)?(an?\s+|both\s+|all\s+)?([\w'-]+\s+){0,3}?" + RACE_TERMS + r"\b", re.I)
_BEFORE = re.compile(RACE_TERMS + r"(\s+[\w'-]+){0,3}\s*(of\s+)?(the\s+)?$", re.I)


def attached_race(sentence, refs):
    """True if a race term is attached to a reference to the plaintiff(s): "Plaintiff is Black", "plaintiffs,
    who are blacks", "Smith, an African American man", "I am a black man", "Plaintiff is not white",
    "the racial minority of Plaintiff X", "a young African American male Plaintiff Y"."""
    for rx in refs:
        for m in rx.finditer(sentence):
            if _AFTER.search(sentence[m.end():m.end() + 90]) or _BEFORE.search(sentence[max(0, m.start() - 70):m.start()]):
                return True
    return False


def name_anchors_21(name, n_plaintiffs, pro_se, texts):
    """Like name_anchors, but when someone else in the complaint shares the plaintiff's surname (a decedent,
    a relative, a co-plaintiff), the bare surname is ambiguous: only the full name counts as a name reference."""
    anchors = name_anchors(name, n_plaintiffs, pro_se)
    parts = [p for p in re.split(r"[ ,]+", str(name)) if p and not re.fullmatch(r"(jr|sr|ii|iii|iv)\.?", p, re.I)]
    if len(parts) < 2 or len(parts[-1]) <= 2:
        return anchors
    first, last = parts[0].lower().strip("."), parts[-1]
    full = " ".join(texts)
    others = {m.group(1).lower() for m in re.finditer(r"\b([A-Z][a-z]+)\s+(?:[A-Z]\.?\s+|[A-Z][a-z]+\s+)?" + re.escape(last) + r"\b", full)}
    others -= {first, "mr", "ms", "mrs", "miss", "dr", "plaintiff", "plaintiffs", "officer", "defendant", "the", "and",
               "of", "v", "vs", "decedent", "sgt", "detective", "deputy", "judge", "chief"}
    if not others:
        return anchors
    full_rx = re.compile(re.escape(re.sub(r"\s+", " ", str(name)).strip()), re.I)
    return [(0, full_rx)] + [(p, rx) for p, rx in anchors if p != 0]


def page_texts(pdf_path, max_pages=60):
    with fitz.open(pdf_path) as d:
        return [re.sub(r"\s+", " ", HEADER_RE_21.sub(" ", d[i].get_text())) for i in range(min(max_pages, len(d)))]


def candidates_21(sents, texts, anchors, n_plaintiffs, limit=20):
    """Race candidates: sentences with a reference to the plaintiff (or 'Plaintiffs') and a race cue, ranked so
    that sentences where the race term is attached to the reference come first; a larger limit for long
    filings; plus 'Race: X' fields on pages that name the plaintiff (forms and court records)."""
    refs = [rx for _, rx in anchors] + [GROUP_RE]     # "plaintiffs, who are …" also in single-plaintiff templates
    name_rx = [rx for p, rx in anchors if p == 0]
    lim = 30 if len(texts) > 30 else limit
    found = []
    for k, (pg, s) in enumerate(sents):
        if not RACE_CUES_21.search(s):
            continue
        best = None
        for prio, rx in anchors:
            if rx.search(s):
                d = _distance(s, rx, RACE_CUES_21)
                if best is None or (prio, d) < best:
                    best = (prio, d)
        is_group = bool(GROUP_RE.search(s))
        if best is None and is_group and (n_plaintiffs > 1 or attached_race(s, [GROUP_RE])):
            best = (1, _distance(s, GROUP_RE, RACE_CUES_21))
        if best is None:
            continue
        if attached_race(s, refs):
            best = (-1, best[1])
        prev = sents[k - 1][1] if k > 0 and sents[k - 1][0] == pg else ""
        found.append({"page": pg, "sentence": s, "context": prev, "group": bool(is_group), "form": False,
                      "priority": best[0], "distance": best[1]})
    for i, t in enumerate(texts):
        if any(rx.search(t) for rx in name_rx):
            for m in FORM_RE.finditer(t):
                frag = t[max(0, m.start() - 120):m.end() + 40]
                found.append({"page": i + 1, "sentence": frag, "context": "", "group": False, "form": True,
                              "priority": -1, "distance": 0})
    out = []
    for prio in (-1, 0, 1, 2):
        grp = sorted([c for c in found if c["priority"] == prio], key=lambda c: (c["distance"], c["page"]))
        out += grp[:lim // 2] if prio >= 0 else grp[:lim]
    return out[:lim]


SYSTEM_PROMPT_21 = """You code a plaintiff's RACE from sentences of a civil-rights complaint.
Rules:
- Use ONLY the sentences given. Report a race only if a sentence states or clearly indicates it FOR THIS PLAINTIFF.
- "explicit": a race or ethnicity describes the plaintiff ("Plaintiff is an African-American male", "I am a black man", "Plaintiffs are Black", "minority groups such as plaintiffs, who are blacks", a form or court record that names the plaintiff and lists "Race: Black").
- A group statement about ALL plaintiffs applies to this plaintiff, even inside a longer sentence about a population, if it says the plaintiffs are of that race. A statement about a subset ("the plaintiffs who were arrested") applies only if this plaintiff is in it.
- "racial minority of Plaintiff X", "Plaintiff is not white", "non-white", "a person of color" describing the plaintiff: category "Person of color (unspecified)".
%s- NOT a race: statements about populations ("the rights of African Americans", "the Black residents of the county"), an allegation of discrimination "based on race" without naming the plaintiff's race, descriptions of other people (officers, suspects, decedents, relatives, other cases), names, places, languages, or national origins (except Latin American origins, which indicate Hispanic/Latino).
- In a first-person (pro se) complaint, "I"/"my" refers to the plaintiff.
- The quote must be copied EXACTLY from one sentence (no more than 40 words).
Return JSON only:
{"race": {"value": "as written or not stated", "category": "<one of: %s>|not stated", "basis": "explicit|inferred|", "quote": "", "page": 0}}"""
_INDIRECT = ('- "inferred": an indirect statement about the plaintiff\'s own race: "Plaintiff did nothing other than drive '
             'while black", a racial slur aimed at the plaintiff, "if I were a white individual" (= Person of color).\n')
SYSTEM_PROMPT_21 = SYSTEM_PROMPT_21 % (_INDIRECT if INCLUDE_INDIRECT else "", ", ".join(RACE_CATEGORIES))


def race_category_21(value, category):
    v = str(value).lower()
    if re.search(r"\bminority\b|not white|non-?white|of color", v):
        return "Person of color (unspecified)"
    return race_category(value, category)


def verify_21(answer, cands, anchors, n_plaintiffs):
    """Race only: the quote must be one of the candidates (citation check); a group statement must cover all
    plaintiffs; a population statement is rejected unless its race term is attached to the plaintiff(s); a
    form field counts when its page names the plaintiff."""
    a = (answer or {}).get("race") or {}
    val = str(a.get("value", "")).strip()
    if not val or val.lower() == "not stated":
        return None
    quote = str(a.get("quote", ""))
    q = norm_text(quote)
    match = [c for c in cands if len(q) >= 8 and q in (norm_text(c["sentence"]) + " " + norm_text(c["context"]))]
    if not match:
        return {"rejected": True, "reason": "quote not found in the candidate sentences", **a}
    if any(c["form"] for c in match):
        return a
    refs = [rx for _, rx in anchors] + [GROUP_RE]
    name_rx = [rx for p, rx in anchors if p == 0]
    names_plaintiff = any(rx.search(quote) for rx in name_rx)
    if n_plaintiffs > 1 and not names_plaintiff and GROUP_RE.search(quote):
        if SUBSET_RE.search(quote) or (re.search(r"\bboth\b", quote, re.I) and n_plaintiffs != 2):
            return {"rejected": True, "reason": "group statement limited to some plaintiffs", **a}
    if not RACE_CUES_21.search(quote) and not (INCLUDE_INDIRECT and re.search(r"\bnigg|\bwhile black", quote, re.I)):
        return {"rejected": True, "reason": "quote states no race", **a}
    attached = attached_race(quote, refs)
    indirect = INCLUDE_INDIRECT and str(a.get("basis", "")).lower() == "inferred" and \
        (re.search(r"while black|if (i|he|she|plaintiff) were (a )?white", quote, re.I) or re.search(r"\bnigg", quote, re.I))
    if not attached and not indirect:
        if RACE_GENERAL_RE.search(quote):
            return {"rejected": True, "reason": "describes a population, not the plaintiff", **a}
        if QUOTED_RE.search(quote):
            return {"rejected": True, "reason": "race term inside a quoted title or remark", **a}
        if not (names_plaintiff or PLAINTIFF_REF_RE.search(quote)):
            return {"rejected": True, "reason": "quote does not refer to the plaintiff", **a}
    return a


def ask_model_21(name, cands, n_plaintiffs, pro_se):
    lines = []
    for c in cands:
        ctx = f"(previous sentence: {c['context'][:300]}) " if c["context"] else ""
        tag = "[form or record] " if c["form"] else ""
        lines.append(f"[p. {c['page']}] {tag}{ctx}{c['sentence']}")
    user = (f"Plaintiff: {name}\nNumber of plaintiffs in the complaint: {n_plaintiffs}\n"
            f"Self-written (pro se) complaint: {'yes' if pro_se else 'no'}\n\nSentences:\n" + "\n".join(lines))
    resp = client.chat.completions.create(
        model=SECOND_PASS_MODEL, temperature=0, response_format={"type": "json_object"},
        messages=[{"role": "system", "content": SYSTEM_PROMPT_21}, {"role": "user", "content": user}])
    return json.loads(resp.choices[0].message.content)


def race_pass_21(plaintiffs, race_col, folder, cache_path, workers=4, limit=None):
    """Version 2.1 race pass for individual plaintiffs whose race is still missing in `race_col` (all of them,
    or the first `limit` for a test)."""
    cache = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
    lock = threading.Lock()
    todo = plaintiffs[(plaintiffs.plaintiff_type == "person") & (plaintiffs.exclude.fillna("") == "") &
                      plaintiffs[race_col].isna()]
    if limit:
        todo = todo.head(limit)
    n_by_file = plaintiffs.groupby("source_file").size()
    text_cache = {}

    def work(row):
        path = os.path.join(folder, row["source_file"])
        if not os.path.exists(path):
            return {"status": "file not found"}
        with lock:
            have = row["source_file"] in text_cache
        if not have:
            st, tx = page_sentences(path), page_texts(path)
            with lock:
                text_cache[row["source_file"]] = (st, tx)
        sents, texts = text_cache[row["source_file"]]
        n_pl = int(n_by_file.get(row["source_file"], 1))
        anchors = name_anchors_21(row["name_final"], n_pl, bool(row["pro_se"]), texts)
        cands = candidates_21(sents, texts, anchors, n_pl)
        if not cands:
            return {"status": "no candidate sentences"}
        key = hashlib.md5(f"{row['source_file']}|{row['name_final']}|race|v21".encode()).hexdigest()
        with lock:
            answer = cache.get(key)
        if answer is None:
            answer = ask_model_21(row["name_final"], cands, n_pl, bool(row["pro_se"]))
            with lock:
                cache[key] = answer
        a = verify_21(answer, cands, anchors, n_pl)
        if a is None:
            return {"status": "not stated"}
        if a.get("rejected"):
            return {"status": f"rejected: {a['reason']}", "value": a.get("value"), "quote": a.get("quote"),
                    "page": a.get("page")}
        return {"status": "found", "value": a.get("value"), "category": race_category_21(a.get("value"), a.get("category")),
                "basis": a.get("basis", ""), "quote": a.get("quote"), "page": a.get("page")}

    rows = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(work, r): r for _, r in todo.iterrows()}
        for k, fut in enumerate(as_completed(futures), start=1):
            r = futures[fut]
            try:
                res = fut.result()
            except Exception as e:
                res = {"status": f"error: {str(e)[:120]}"}
            rows.append({"source_file": r["source_file"], "name": r["name_final"], "pro_se": r["pro_se"],
                         "variable": "race", **res})
            if k % 25 == 0:
                with lock:
                    json.dump(cache, open(cache_path, "w"))
                print(f"  {k}/{len(futures)} plaintiffs")
    json.dump(cache, open(cache_path, "w"))
    return pd.DataFrame(rows)


def partly_scanned(folder, min_share=0.5, min_words=50):
    """Complaints where most pages carry no readable text once the court's header stamp is removed (pages
    with fewer than `min_words` words), although some pages do: partly scanned files the first OCR screen,
    which averaged the first five pages, could miss."""
    rows = []
    for p in sorted(glob.glob(os.path.join(folder, "*.pdf"))):
        try:
            with fitz.open(p) as d:
                words = [len(HEADER_RE_21.sub(" ", pg.get_text()).split()) for pg in d]
        except Exception:
            continue
        share = sum(w < min_words for w in words) / max(len(words), 1)
        if share >= min_share:
            rows.append({"file": os.path.basename(p), "pages": len(words), "pages_without_text": round(share, 2)})
    return pd.DataFrame(rows)


def ocr_files(folder, files, backup_dir):
    """OCR the given complaints in place (originals kept in backup_dir), with OCRmyPDF run from this Python."""
    if shutil.which("tesseract") is None or shutil.which("gs") is None:
        subprocess.run("apt-get -qq update && apt-get -qq install -y tesseract-ocr ghostscript > /dev/null",
                       shell=True, check=True)
    if importlib.util.find_spec("ocrmypdf") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ocrmypdf"], check=True)
    os.makedirs(backup_dir, exist_ok=True)
    done = []
    for name in files:
        p = os.path.join(folder, name)
        if not os.path.exists(os.path.join(backup_dir, name)):
            shutil.copy2(p, os.path.join(backup_dir, name))
        out = p + ".ocr.pdf"
        r = subprocess.run([sys.executable, "-m", "ocrmypdf", "--force-ocr", "-l", "eng", "--quiet", p, out],
                           capture_output=True, text=True)
        if r.returncode == 0:
            os.replace(out, p)
            done.append(name)
        else:
            print("OCR failed:", name, r.stderr[-200:])
    return done


def draw_samples_21(p21, checked_before, n_represented=70, n_pro_se=15, n_precision=20, seed=2028):
    """Fresh race samples for version 2.1: (a) recall: individual plaintiffs still without a race, excluding all
    plaintiffs checked before; (b) precision: race values added by the version-2.1 pass."""
    people = p21[(p21.plaintiff_type == "person") & (p21.exclude.fillna("") == "")].copy()
    people = people[~(people.source_file + "|" + people.name_final.astype(str)).isin(checked_before)]
    rows = []
    still = people[people.race_v21.isna()]
    for ps, n in ((True, n_pro_se), (False, n_represented)):
        g = still[still.pro_se == ps]
        for _, r in g.sample(min(n, len(g)), random_state=seed).iterrows():
            rows.append({"check": "recall", "variable": "race", "case_name": r.case_name, "source_file": r.source_file,
                         "name": r.name_final, "pro_se": ps, "value_v21": "", "evidence_v21": "",
                         "stated_in_complaint": "", "value_correct": "", "correct_value": "", "note": ""})
    added = people[people.race_v21_source.fillna("").str.startswith("second pass 2.1")]
    for _, r in added.sample(min(n_precision, len(added)), random_state=seed).iterrows():
        rows.append({"check": "precision", "variable": "race", "case_name": r.case_name, "source_file": r.source_file,
                     "name": r.name_final, "pro_se": r.pro_se, "value_v21": r.race_v21, "evidence_v21": r.race_v21_evidence,
                     "stated_in_complaint": "", "value_correct": "", "correct_value": "", "note": ""})
    return pd.DataFrame(rows)


def evaluate_21(p21, sample, recall_before):
    """Race recall after version 2.1 (stratified by pro se, with a 95% CI) and the precision of the 2.1
    additions. Values found automatically: version 1, the version-2 second pass, and the 2.1 pass; values
    added by hand in earlier checks count as misses of the automated pipeline."""
    people = p21[(p21.plaintiff_type == "person") & (p21.exclude.fillna("") == "")]
    v2src = people.race_final_v2_source.fillna("")
    auto_mask = people.race_pipeline.notna() | v2src.str.startswith("second pass") | \
        people.race_v21_source.fillna("").str.startswith("second pass 2.1")
    auto = int(auto_mask.sum())
    manual = int((people.race_v21.notna() & ~auto_mask).sum())
    s = sample.fillna("")
    yn = lambda col: s[col].str.strip().str.lower().isin(["yes", "no"])
    rec = s[(s.check == "recall") & yn("stated_in_complaint")]
    est, est_lo, est_hi, strata = float(manual), float(manual), float(manual), []
    for ps in (True, False):
        sub = rec[rec.pro_se.astype(str).str.lower() == str(ps).lower()]
        n, k = len(sub), int((sub.stated_in_complaint.str.lower() == "yes").sum())
        N = int((people.race_v21.isna() & (people.pro_se == ps)).sum())
        if n:
            plo_s, phi_s = wilson(k, n)          # Wilson bounds per stratum (sensible even when k = 0)
            est += N * k / n
            est_lo += N * plo_s
            est_hi += N * phi_s
        strata.append(f"{'pro se' if ps else 'represented'}: {k} of {n} stated (of {N} still not stated)")
    recall = auto / (auto + est)
    lo, hi = auto / (auto + est_hi), auto / (auto + est_lo)
    pre = s[(s.check == "precision") & yn("value_correct")]
    kp = int((pre.value_correct.str.lower() == "yes").sum())
    plo, phi = wilson(kp, len(pre))
    return pd.DataFrame([{
        "variable": "race", "recall_v2": recall_before, "found_automatically": auto, "added_by_hand_earlier": manual,
        "recall_sample": "; ".join(strata), "estimated_missed": round(est), "recall_v21": round(recall, 2),
        "recall_v21_ci": f"{lo:.0%}-{hi:.0%}",
        "precision_v21_additions": f"{kp} of {len(pre)}" + (f" ({kp/len(pre):.0%}; 95% CI {plo:.0%}-{phi:.0%})" if len(pre) else "")}])

## Step 9: Per-Page OCR Screen (Partly Scanned Complaints)

Lists complaints where at least half the pages carry no readable text once the court's header stamp is removed. The files OCR'd earlier have a text layer now, so they don't reappear; handwritten forms may, and OCR helps them little. Set `RUN_OCR = True` and re-run to OCR the listed files (originals are kept in `scanned_originals/`; about 10 seconds a file). The version-2.1 pass then reads the new text.

*Version-1 values for these files are not re-extracted here; to re-extract them fully, delete their results in `complaints/_results/` and run the extraction notebook's last cell.*

In [ ]:
RUN_OCR = False
scan = partly_scanned(COMPLAINT_FOLDER)
print(f"{len(scan)} partly scanned complaint(s)")
display(scan)
if RUN_OCR and len(scan):
    done = ocr_files(COMPLAINT_FOLDER, scan.file.tolist(), os.path.join(SORTED, "scanned_originals"))
    print(f"OCR done: {len(done)} of {len(scan)}")

## Step 10: Test the Race Pass on 10 Plaintiffs

Check that the values and quotes make sense before the full run.

In [ ]:
CACHE_21 = os.path.join(OUT_DIR, "second_pass_cache_v21.json")
REVIEW_21 = os.path.join(OUT_DIR, "race_review_v21.xlsx")
RESULTS_V21 = os.path.join(OUT_DIR, "final_results_v21.xlsx")
SAMPLE_21 = os.path.join(OUT_DIR, "evaluation_sample_v21.xlsx")
RECALL_V2_RACE = 0.81

test21 = race_pass_21(pf, "race_final_v2", COMPLAINT_FOLDER, CACHE_21, workers=WORKERS, limit=10)
display(test21.reindex(columns=["name", "status", "value", "category", "basis", "quote", "page"]))

## Step 11: The Full Race Pass

In [ ]:
results21 = race_pass_21(pf, "race_final_v2", COMPLAINT_FOLDER, CACHE_21, workers=WORKERS)
print(results21.status.value_counts().to_string())
results21.to_csv(os.path.join(OUT_DIR, "race_pass_v21_results.csv"), index=False)

## Step 12: Review the New Race Values

Writes `race_review_v21.xlsx`: every value found, with its quote and page. Set `accept` to **no** for any value that is wrong; your decisions are kept if the cell is run again.

In [ ]:
found21 = results21[results21.status == "found"].copy()
cols21 = ["source_file", "name", "pro_se", "value", "category", "basis", "quote", "page", "accept"]
found21["accept"] = "yes"
found21 = found21.reindex(columns=cols21)
if os.path.exists(REVIEW_21):
    old = pd.read_excel(REVIEW_21, dtype=str).fillna("")
    key = lambda d: d.source_file.astype(str) + "|" + d["name"].astype(str)
    new = found21[~key(found21).isin(set(key(old)))]
    pd.concat([old, new.astype(str)], ignore_index=True).to_excel(REVIEW_21, index=False)
    print(f"Review file kept ({len(old)} rows); {len(new)} new values added -> {REVIEW_21}")
else:
    found21.to_excel(REVIEW_21, index=False)
    print(f"{len(found21)} new race values -> {REVIEW_21}")
display(found21.category.value_counts())

## Step 13: Build the Version-2.1 Dataset

Starts from Step 8's final dataset; race values marked *yes* in the review fill gaps only, with their source and evidence (`race_v21`).

In [ ]:
rev21 = fix_names(pd.read_excel(REVIEW_21, dtype=str).fillna(""), "name")
acc21 = rev21[rev21.accept.str.lower() == "yes"]
p21 = pf.copy()
p21["race_v21"], p21["race_v21_source"], p21["race_v21_evidence"] = p21["race_final_v2"], p21["race_final_v2_source"], None
for _, r in acc21.iterrows():
    ix = p21.index[(p21.source_file == r.source_file) & (p21.name_final == r["name"]) & p21.race_final_v2.isna()]
    cat = r.category if r.category in RACE_CATEGORIES else race_category_21(r.value, r.category)
    p21.loc[ix, ["race_v21", "race_v21_source", "race_v21_evidence"]] = \
        [cat, f"second pass 2.1 ({r.basis or 'n/a'}; as written: {r.value})", f'p. {r.page}: "{r.quote}"']
people21 = p21[(p21.plaintiff_type == "person") & (p21.exclude.fillna("") == "")]
stated21 = pd.DataFrame([{"variable": "race", "stated_v2_final": round(people21.race_final_v2.notna().mean(), 3),
                          "stated_v21": round(people21.race_v21.notna().mean(), 3),
                          "stated_v21_represented": round(people21[people21.pro_se == False].race_v21.notna().mean(), 3),
                          "stated_v21_pro_se": round(people21[people21.pro_se == True].race_v21.notna().mean(), 3)}])
dist21 = people21.race_v21.fillna("not stated").value_counts().rename_axis("race_v21").reset_index(name="plaintiffs")
with pd.ExcelWriter(RESULTS_V21) as w:
    stated21.to_excel(w, sheet_name="Summary v2.1", index=False)
    dist21.to_excel(w, sheet_name="Summary v2.1", index=False, startrow=4)
    p21.to_excel(w, sheet_name="Plaintiffs v2.1", index=False)
    results21.to_excel(w, sheet_name="Race pass v2.1", index=False)
print(f"race values added: {int(p21.race_v21_source.fillna('').str.startswith('second pass 2.1').sum())} | saved: {RESULTS_V21}")
display(stated21)
display(dist21)

## Step 14: Fresh Race Evaluation Sample

Writes `evaluation_sample_v21.xlsx` (never overwritten): **recall rows**, 70 represented and 15 pro se plaintiffs still without a race after version 2.1 (a larger represented sample, so the interval is narrower), excluding everyone checked in earlier samples; and **precision rows**, 20 race values added by version 2.1. Fill in as before: recall rows `stated_in_complaint` (and the value and quote if yes), precision rows `value_correct`.

In [ ]:
checked = set(qc.source_file + "|" + qc.name_clean)
prev = pd.read_excel(SAMPLE, dtype=str).fillna("")
checked |= set(prev.source_file + "|" + prev["name"])
if os.path.exists(SAMPLE_21):
    print("Sample exists and was NOT overwritten:", SAMPLE_21)
else:
    s21 = draw_samples_21(p21, checked)
    s21.to_excel(SAMPLE_21, index=False)
    print(s21.groupby(["check", "pro_se"]).size().to_string(), "->", SAMPLE_21)

## Step 15: Race Recall Before and After Version 2.1, and the Final Dataset

Run after filling in the version-2.1 sample. Values you find in recall rows are added to the final dataset (source: *evaluation, by hand*), and rejected precision values are removed; saves `final_results_v21_final.xlsx`.

In [ ]:
s21 = fix_names(pd.read_excel(SAMPLE_21, dtype=str).fillna(""), "name")
ev21 = evaluate_21(p21, s21, RECALL_V2_RACE)
display(ev21)
r = ev21.iloc[0]
print(f"race: estimated recall {r.recall_v2:.0%} (version 2) -> {r.recall_v21:.0%} (version 2.1; 95% CI {r.recall_v21_ci}); "
      f"precision of the 2.1 additions {r.precision_v21_additions}")

pfinal = p21.copy()
pfinal["race_final_v21"], pfinal["race_final_v21_source"] = pfinal["race_v21"], pfinal["race_v21_source"]
for _, x in s21.iterrows():
    ix = pfinal.index[(pfinal.source_file == x.source_file) & (pfinal.name_final == x["name"])]
    if x.check == "recall" and x.stated_in_complaint.strip().lower() == "yes" and x.correct_value.strip():
        pfinal.loc[ix, ["race_final_v21", "race_final_v21_source"]] = \
            [race_category_21(x.correct_value, x.correct_value), "evaluation, by hand"]
    elif x.check == "precision" and x.value_correct.strip().lower() == "no":
        cv = x.correct_value.strip()
        pfinal.loc[ix, ["race_final_v21", "race_final_v21_source"]] = \
            [None if not cv or cv.lower() == "not stated" else race_category_21(cv, cv),
             "evaluation, by hand (2.1 value rejected)"]
pp = pfinal[(pfinal.plaintiff_type == "person") & (pfinal.exclude.fillna("") == "")]
dist_f = pp.race_final_v21.fillna("not stated").value_counts().rename_axis("race").reset_index(name="plaintiffs")
st = dist_f.loc[dist_f.race != "not stated", "plaintiffs"].sum()
dist_f["share_of_stated"] = [round(n / st, 3) if c != "not stated" else None for c, n in zip(dist_f.race, dist_f.plaintiffs)]
FINAL_21 = os.path.join(OUT_DIR, "final_results_v21_final.xlsx")
with pd.ExcelWriter(FINAL_21) as w:
    dist_f.to_excel(w, sheet_name="Summary", index=False)
    ev21.to_excel(w, sheet_name="Evaluation v2.1", index=False)
    pfinal.to_excel(w, sheet_name="Plaintiffs (final)", index=False)
print(f"race stated: {pp.race_final_v21.notna().mean():.1%} of {len(pp)} individual plaintiffs | saved: {FINAL_21}")
display(dist_f)

## Step 15b: F1 and Accuracy for Race (Version 2.1)

The same measures for race after version 2.1. The precision of the values found before version 2.1 (version 1 and the version-2 additions) is held at its earlier estimate; the 2.1 additions use the new precision sample.

In [ ]:
people_21 = p21[(p21.plaintiff_type == "person") & (p21.exclude.fillna("") == "")]
v2_added = people_21.race_final_v2_source.fillna("").str.startswith("second pass")
v21_added = people_21.race_v21_source.fillna("").str.startswith("second pass 2.1")
auto_21 = people_21.race_pipeline.notna() | v2_added | v21_added
n_v1r, n_v2r = int((people_21.race_pipeline.notna()).sum()), int((v2_added & people_21.race_pipeline.isna()).sum())
p_v2_add = 1.0   # precision of the version-2 race additions (evaluation sample: 20 of 20)
p_before = (n_v1r * PRECISION_V1["race"] + n_v2r * p_v2_add) / max(n_v1r + n_v2r, 1)
metrics_v21 = pd.DataFrame([overall_metrics(people_21, "race", auto_21, v21_added, "race_v21", s21, "value_correct",
                                            p_v1=p_before)])
display(pd.concat([metrics_v2[metrics_v2.variable == "race"].assign(version="2"),
                   metrics_v21.assign(version="2.1")], ignore_index=True))
with pd.ExcelWriter(FINAL_21, mode="a", engine="openpyxl", if_sheet_exists="replace") as w:
    metrics_v21.to_excel(w, sheet_name="F1 and accuracy v2.1", index=False)

## Step 16: Age Precision Check

Age was checked only on the gold set (5 of 5 correct). This step estimates the precision of the reported ages on a random sample of 30 of the 99 stated ages; no model calls.

**Can be run on its own:** after a restart, run Setup, Settings and the two Functions cells (they only define functions), then the two cells below. The steps with model calls are not needed.

1. Run the first cell: it writes `age_precision_sample.xlsx` to the version-2 folder. If the file is already there, it is kept with your answers.
2. Fill in `value_correct` (yes / no) for each row, using the page and quote, and `correct_age` where the answer is no.
3. Run the second cell: precision with a 95% Wilson interval, saved as the "Age precision" sheet of `final_results_v21_final.xlsx`.

**Result (version 2.1):** 30 of 30 ages correct, precision **100% (95% CI 89–100%)**.

Recall for age is not estimated here: age is stated for only about 15% of plaintiffs, so a random sample of "not stated" values would find too few misses to measure. A full-text scan for age phrasings is planned for the next iteration.


In [ ]:
# ---- Age precision check: a random sample of the stated ages, with the pipeline's quote and page ---------
FINAL_21 = os.path.join(OUT_DIR, "final_results_v21_final.xlsx")   # also set in Step 15; repeated so this step runs on its own
AGE_SAMPLE = os.path.join(OUT_DIR, "age_precision_sample.xlsx")
AGE_N, AGE_SEED = 30, 2026

def draw_age_sample(final_file, library_file, n=AGE_N, seed=AGE_SEED):
    f = pd.read_excel(final_file, sheet_name="Plaintiffs (final)")
    f = f[(f.plaintiff_type == "person") & f.exclude.isna() & f.age_final.notna()].copy()
    ev = pd.read_excel(library_file, sheet_name="evidence")
    ev = ev[ev.field.str.match(r"plaintiffs\[\d+\]\.age$")].copy()
    ev["age_final"] = pd.to_numeric(ev.value, errors="coerce")
    ev = ev.dropna(subset=["quote"]).drop_duplicates(["source_file", "age_final"])
    f = f.merge(ev[["source_file", "age_final", "page", "quote", "status"]], on=["source_file", "age_final"], how="left")
    f = f.sort_values(["source_file", "name_final"]).reset_index(drop=True)
    s = f.sample(n=min(n, len(f)), random_state=seed).sort_values(["source_file", "name_final"])
    out = pd.DataFrame({
        "case_name": s.case_name, "source_file": s.source_file, "name": s.name_final,
        "age_reported": s.age_final.astype(int), "page": s.page, "quote": s.quote,
        "citation_check": s.status.fillna("added by hand"),
        "value_correct": "", "correct_age": "", "note": ""})
    return out, len(f)

if not os.path.exists(AGE_SAMPLE):
    print("No filled sample found under the expected name; drawing a new one.")
    age_sample, n_stated = draw_age_sample(FINAL_21, LIBRARY)
    age_sample.to_excel(AGE_SAMPLE, index=False)
    print(f"{len(age_sample)} of {n_stated} stated ages sampled -> {AGE_SAMPLE}")
else:
    print("Sample already exists (kept, with your answers):", AGE_SAMPLE)
    others = [f for f in os.listdir(OUT_DIR) if f.startswith("age_precision_sample") and f != os.path.basename(AGE_SAMPLE)]
    if others:
        print("Note: other copies in the folder, not used:", others)


In [ ]:
# ---- Age precision: results ------------------------------------------------------------------------------
age_s = pd.read_excel(AGE_SAMPLE, dtype=str).fillna("")
ans = age_s.value_correct.str.strip().str.lower()
k, n = int((ans == "yes").sum()), int(ans.isin(["yes", "no"]).sum())
if n < len(age_s):
    print(f"Note: {len(age_s) - n} rows not filled in yet.")
lo, hi = wilson(k, n)
age_metrics = pd.DataFrame([{"variable": "age", "precision": round(k / n, 3) if n else None,
                             "precision_95ci": f"{lo:.0%}-{hi:.0%}" if n else "", "correct": k, "checked": n,
                             "stated_ages": 99, "recall": "not estimated"}])
display(age_metrics)
wrong = age_s[ans == "no"]
if len(wrong):
    print("Ages to correct in the final dataset:")
    display(wrong[["case_name", "name", "age_reported", "correct_age", "note"]])
with pd.ExcelWriter(FINAL_21, mode="a", engine="openpyxl", if_sheet_exists="replace") as w:
    age_metrics.to_excel(w, sheet_name="Age precision", index=False)


## Notes

- **Version 1 is unchanged** and remains the validated baseline; version 2 is reported as an improvement on top of it, measured on a fresh sample.
- **The setting recovery** restores values whose quote was verified on the cited page; they are marked *for review*, since the cue-word check did not confirm them. The location variables remain experimental until validated on new complaints.
- **Re-running** is free: model answers are cached in `second_pass_cache.json`. Delete the cache to start over.
- **Version 2.1** changes race only, and keeps its own cache (`second_pass_cache_v21.json`), review file and evaluation sample; the version-2 files are not modified.